- Thanks to [@Dayuxiaoshui](https://github.com/Dayuxiaoshui) for providing this fine-tuning recipe.

Licensed under the MIT License (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://opensource.org/licenses/MIT

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Full-Duplex Voice Dialogue: LoRA Fine-tuning Ling-3.0-tiny for Turn-Taking

<table align="left">
  <td>
    <a target="_blank" href="https://www.ant-ling.com/"><img src="https://img.shields.io/badge/Official_Website-ant--ling.com-6366f1?style=flat-square" alt="Website" /></a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/inclusionAI/ling-cookbook"><img src="https://img.shields.io/badge/GitHub-Ling_Cookbook-181717?style=flat-square&logo=github" alt="GitHub" /></a>
  </td>
  <td>
    <a target="_blank" href="https://huggingface.co/inclusionAI"><img src="https://img.shields.io/badge/Hugging_Face-inclusionAI-FFD21E?style=flat-square&logo=huggingface&logoColor=black" alt="Hugging Face" /></a>
  </td>
  <td>
    <a target="_blank" href="https://www.modelscope.cn/organization/inclusionAI"><img src="https://img.shields.io/badge/ModelScope-inclusionAI-624AFF?style=flat-square" alt="ModelScope" /></a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/inclusionAI/AReno"><img src="https://img.shields.io/badge/GitHub-AReno-181717?style=flat-square&logo=github" alt="AReno" /></a>
  </td>
</table>

<br><br>

## 1. What This Task Does

A full-duplex voice assistant processes incoming speech and generates output simultaneously. Beyond generating reply content, the system must continuously decide which turn-taking action to take: continue waiting during a mid-sentence pause, emit a brief acknowledgment during a long user monologue, respond when the user finishes a turn, stop speaking when interrupted, and ignore speech not directed at the assistant.

This recipe fine-tunes the 7.9B-total, 1.3B-active MoE model [Ling-3.0-tiny](https://huggingface.co/inclusionAI/Ling-3.0-tiny) with LoRA into a turn-taking decision and response generation model. At every decision moment it reads the live ASR text and outputs one action tag, followed by a short reply when responding or acknowledging:

| Action | When |
| :--- | :--- |
| LISTEN | The user has a mid-sentence pause, hesitates, or has not finished speaking. Stay silent. |
| BACKCHANNEL | The user is in a long turn. Give a short acknowledgment without interrupting the turn. |
| SPEAK | The user finished a turn addressed to the assistant. Reply. |
| STOP | The assistant is speaking and the user interrupts with a new or revised request. Stop and respond. |
| IGNORE | Brief acknowledgments, the user talking to someone else, or background speech that needs no response. |

Evaluated on decision points built from [Full-Duplex-Bench](https://github.com/DanielLin94144/Full-Duplex-Bench) v1.0 / v1.5, the best checkpoint achieves the following improvements:

- Accuracy increases from 0.581 to 0.885
- Macro F1 increases from 0.374 to 0.737
- False response rate on speech that should be ignored drops from 33.6% to 0.7%
- Interruption stop rate increases from 62.5% to 96.5%

No Full-Duplex-Bench data is used during training.

## 2. Important Notes

Before running or adapting the training and evaluation workflow in this recipe, please note the following task boundaries and experimental limitations:

- The model only receives ASR text and a fixed silence duration, and does not learn from pause length, prosody, or raw audio. In a production system it should sit downstream of a VAD or endpointer that decides when to invoke it.
- The evaluation metrics are computed on text-level decision points derived from Full-Duplex-Bench annotations rather than official audio-level metrics, and should not be compared directly with end-to-end speech-to-speech leaderboard scores.
- The released r3 checkpoint at step 800 was selected based on Full-Duplex-Bench accuracy, so its metrics are slightly optimistic; r1 and r2 use their final training checkpoints.
- The training set consists entirely of synthetic English examples generated by the base model itself, so topic and phrasing diversity is limited, such as relatively uniform phrasings when opening a topic switch.
- The model rarely emits BACKCHANNEL. If your application requires active backchannels, increase `P_BACKCHANNEL` in `data/build_train.py` or use the r2 checkpoint.

## 3. Model and Fine-Tuning Framework

This recipe builds on Ling-3.0-tiny and the [AReno](https://github.com/inclusionAI/AReno) post-training and serving framework. The base model, fine-tuned LoRA adapter weights, training dataset, and evaluation source are summarized below:

| Artifact | Link |
| :--- | :--- |
| LoRA adapter weights, PEFT format, ≈2.4 GB | [Dayuxiaoshui/Ling-3.0-tiny-Full-Duplex-Brain-LoRA](https://huggingface.co/Dayuxiaoshui/Ling-3.0-tiny-Full-Duplex-Brain-LoRA) |
| Synthetic scenarios and train/validation splits | [Dayuxiaoshui/Ling-3.0-tiny-Full-Duplex-Brain-SFT](https://huggingface.co/datasets/Dayuxiaoshui/Ling-3.0-tiny-Full-Duplex-Brain-SFT) |
| Base model | [inclusionAI/Ling-3.0-tiny](https://huggingface.co/inclusionAI/Ling-3.0-tiny) |
| Evaluation source | [Full-Duplex-Bench](https://github.com/DanielLin94144/Full-Duplex-Bench), converted into `fdb_eval.jsonl` via Step 1 in Section 6 |

## 4. Hardware and Software Requirements

Reproducing the full training and concurrent evaluation workflow uses the following hardware and software environment:

| Item | Version used |
| :--- | :--- |
| GPU | 8× NVIDIA H20 96GB, using 8 GPUs for training and 4 GPUs for serving |
| Training and serving framework | AReno, commit 3cfa480 |
| Base model | inclusionAI/Ling-3.0-tiny |
| Eval data | Full-Duplex-Bench v1.0 and v1.5 text annotations |
| Script dependencies | httpx, transformers |

The 8× H20 setup in the table above reflects the author's original server environment. If you want to run this experiment on consumer hardware, see "[Lightweight Hardware Requirements](#lightweight-hardware)" in the final section.

Set `ARENO_VENV=/path/to/venv` if `areno` is not on your system PATH. Every step below runs from this directory.

In [ ]:
%%bash
export M=/path/to/Ling-3.0-tiny        # local snapshot of inclusionAI/Ling-3.0-tiny
export FDB=/path/to/Full-Duplex-Bench  # directory containing v1.0/ and v1.5/

## 5. Task Analysis

Below we describe the input-output formulation for a single decision moment and the companion source code layout in this directory.

### Task Formulation

One request corresponds to one decision moment. The input consists of the assistant state, the user's ASR text so far, and the silence duration since the user's last word:

```text
Assistant state: speaking (currently answering: "Can you help me plan a weekend trip to Kyoto?")
New user speech: "wait, actually make it Osaka instead"
Silence since the last word: 0.3s
Action:
```

The output begins with an action tag, followed by a reply after SPEAK or STOP, or a short acknowledgment after BACKCHANNEL:

```text
[STOP] No problem, Osaka is a great choice too.
```

The system prompt and input format are defined in [`fdb_brain/prompt.py`](fdb_brain/prompt.py) and shared across data construction, evaluation, and inference. Placing the action tag first allows a downstream controller to act on the first few tokens without waiting for the full reply to finish generating.

Both training and evaluation use a fixed silence duration: 0.8s when the assistant is silent, and 0.3s while it is speaking. The label therefore cannot be inferred from pause duration alone; distinguishing a mid-sentence pause from a completed turn depends on the semantic content of the user's speech, which is the signal a text-only LLM adds on top of a VAD.

### Directory Layout

```text
full-duplex-turn-taking-lora/
├── fdb_brain/prompt.py        # shared system prompt, input rendering, action parser
├── data/
│   ├── build_fdb_eval.py      # Full-Duplex-Bench annotations -> 1862 eval decision points
│   ├── synth_scenarios.py     # generate dialogue scenarios with the base model
│   └── build_train.py         # scenarios -> labelled, pre-tokenized SFT rows
├── train/
│   ├── train_lora.sh          # AReno LoRA SFT (training config)
│   └── sft_loader.py          # dataset loader passed to `areno train`
├── scripts/serve.sh           # start an OpenAI-compatible `areno serve` (optionally with LoRA)
├── eval/run_eval.py           # query + score; writes per-point jsonl and a report json
├── infer.py                   # inference example (single decision or scripted call)
└── results/*.report.json      # evaluation reports of every run in the tables below
```

## 6. Step-by-Step Guide

Follow the steps below to build the evaluation set, generate synthetic training scenarios, expand labelled decision points, run LoRA SFT, serve and evaluate the model, and run inference examples.

### Step 1: Build the evaluation set

In [ ]:
%%bash
python data/build_fdb_eval.py --fdb-root $FDB --out data/fdb_eval.jsonl

Each Full-Duplex-Bench clip is converted into one or more text-level decision points, totaling 1,862 points:

| FDB subset | Decision point | Gold | n |
| :--- | :--- | :--- | ---: |
| v1.0 candor_pause_handling | text up to each annotated PAUSE | LISTEN | 277 |
| v1.0 synthetic_pause_handling | text up to each pause / full turn | LISTEN / SPEAK | 137 / 137 |
| v1.0 icc_backchannel | text up to each ≥0.3s gap in a long monologue | BACKCHANNEL or LISTEN | 315 |
| v1.5 user_interruption | request while silent / interruption while speaking | SPEAK / STOP | 200 / 200 |
| v1.5 user_backchannel | request / brief acknowledgment | SPEAK / IGNORE | 98 / 98 |
| v1.5 talking_to_other | request / user talking to someone else | SPEAK / IGNORE | 100 / 100 |
| v1.5 background_speech | request / background speech | SPEAK / IGNORE | 100 / 100 |

### Step 2: Generate training scenarios with the base model

The training data is fully synthetic. Serve the base model with thinking disabled, then query it to generate dialogue scenarios. Each scenario covers one topic and persona combination and contains 14 fields including a request and reply, an interruption and its reply, side talk, background speech, a long story, and a topic switch, as defined in [`data/synth_scenarios.py`](data/synth_scenarios.py).

In [ ]:
%%bash
bash scripts/serve.sh $M 0,1,2,3 8100 /tmp/serve-base.log --disable-thinking
python data/synth_scenarios.py --base-url http://127.0.0.1:8100 --n 1500 --seed 1 --out data/scenarios_r2.jsonl
python data/synth_scenarios.py --base-url http://127.0.0.1:8100 --n 800  --seed 2 --out data/scenarios_r2b.jsonl

Malformed generations are filtered out, keeping 984 and 498 scenarios from the two runs. The released model also incorporates 1,200 scenarios in `data/scenarios.jsonl` generated with an earlier 8-field prompt. All of these scenarios are included in the released dataset and can be downloaded directly without regenerating.

### Step 3: Expand scenarios into labelled decision points

In [ ]:
%%bash
python data/build_train.py \
  --scenarios data/scenarios.jsonl data/scenarios_r2.jsonl data/scenarios_r2b.jsonl \
  --fdb-eval data/fdb_eval.jsonl --tokenizer $M --out-dir data/

Each scenario expands into up to 18 decision points covering all five actions:

- Request cut mid-sentence → LISTEN
- Complete request → SPEAK + reply
- User interruption while assistant is speaking → STOP + reply
- Side talk while assistant is speaking → IGNORE
- Long story cut after a complete sentence → BACKCHANNEL

The complete expansion rules are documented in [`data/build_train.py`](data/build_train.py). Three design choices are worth noting:

- Adding ASR-style noise: user text is randomly rendered without punctuation and with UM or UH fillers at a 35% probability each, preventing the model from relying solely on sentence-final punctuation.
- Constructing mid-turn hard negatives: two windows are extracted from each long story so that they end on a grammatically complete clause before the story is actually finished, labelled as LISTEN. Without these examples, the model tends to predict SPEAK on any complete clause and interrupt during pauses.
- Preventing evaluation leakage: any synthetic scenario whose text overlaps with a Full-Duplex-Bench evaluation example is removed.

Because AReno's raw-text SFT path does not recognize Ling's role markers and would wrap the prompt twice, the script pre-tokenizes all examples using the exact chat template rendered by `areno serve --disable-thinking`. This produces `train.jsonl` with 32,423 rows (12,521 LISTEN, 7,635 IGNORE, 7,192 SPEAK, 3,951 STOP, and 1,124 BACKCHANNEL) and `val.jsonl` with 1,662 rows.

### Step 4: LoRA SFT

In [ ]:
%%bash
EPOCHS=2 TP=4 SAVE_INTERVAL=200 bash train/train_lora.sh $M data/train.jsonl ckpt/lora-r3 0,1,2,3,4,5,6,7

The training configuration is defined in [`train/train_lora.sh`](train/train_lora.sh):

| Hyper-parameter | Value |
| :--- | :--- |
| LoRA rank / alpha | 32 / 64 across q, k, v, o, gate, up, down projections |
| Batch size / micro batch | 64 / 16 |
| Epochs | 2, totaling 1014 steps |
| LR schedule | 2e-4 → 2e-5, cosine |
| Max prompt / response tokens | 512 / 96 |
| Parallelism | TP=4, DP=2 on 8× H20, ≈4.7 s/step, ≈80 min total |

The training loss is computed only on the action tag and reply tokens, converging to roughly 0.32. AReno's SFT trainer saves checkpoints at intervals of `SAVE_INTERVAL=200`, producing checkpoints from step 200 to step 1000. The released adapter is `step_000800`.

### Step 5: Serve and evaluate

In [ ]:
%%bash
# baseline
bash scripts/serve.sh $M 0,1,2,3 8100 /tmp/serve-base.log --disable-thinking
python eval/run_eval.py --base-url http://127.0.0.1:8100 --data data/fdb_eval.jsonl --out results/baseline_nothink.jsonl

# fine-tuned (pass the LoRA rank explicitly: areno serve defaults to 8)
bash scripts/serve.sh $M 4,5,6,7 8101 /tmp/serve-lora.log --disable-thinking \
  --lora-rank 32 --lora-alpha 64 --lora-adapter-path ckpt/lora-r3/step_000800
python eval/run_eval.py --base-url http://127.0.0.1:8101 --data data/fdb_eval.jsonl --out results/lora_r3_step_000800.jsonl

The evaluation script `eval/run_eval.py` uses greedy decoding with temperature 0 and a maximum of 128 tokens, writing per-example predictions to a jsonl file and summary metrics to the corresponding report.json file. Pass `--score-only` to recompute metrics on an existing prediction file without querying the server again.

### Step 6: Inference example

In [ ]:
%%bash
python infer.py --base-url http://127.0.0.1:8101 --demo

Sample output from the `step_000800` adapter:

```text
[silent  ] user: 'Hey, can you help me plan a'
           -> LISTEN
[silent  ] user: 'Hey, can you help me plan a weekend trip to Kyoto?'
           -> SPEAK  "Sure, I can help you plan a trip to Kyoto. Let's start by picking the dates and your budget."
[speaking] user: 'yeah'
           -> IGNORE
[speaking] user: 'honey, can you grab my keys?'
           -> IGNORE
[speaking] user: 'wait, actually make it Osaka instead'
           -> STOP  'No problem, Osaka is a great choice too.'
[silent  ] user: 'So last time I went to Japan I got completely lost in the subway, and then'
           -> LISTEN
[silent  ] user: 'So last time I went to Japan I got completely lost in the subway, and then a stranger walked me all the way to my hotel.'
           -> SPEAK  "That's a great story, I bet you had a lot of fun getting back on the right train."
```

For a single decision, run `python infer.py --agent-state speaking --history "..." --user "..." --silence 0.3`. Any OpenAI-compatible server that applies Ling's chat template with thinking disabled and loads this PEFT adapter can be called the same way.

## 7. Typical Results

Evaluated on the 1,862 decision points derived from Full-Duplex-Bench, the metric definitions, baseline vs. fine-tuned comparison, per-source breakdown, checkpoint sweep, and behavioral analysis are presented below.

### Metrics

| Metric | Definition | Better |
| :--- | :--- | :---: |
| accuracy | prediction belongs to the accepted label set | ↑ |
| macro F1 | mean F1 over the 5 actions, requiring BACKCHANNEL to be predicted strictly as BACKCHANNEL | ↑ |
| takeover rate | gold LISTEN or BACKCHANNEL, predicted SPEAK or STOP | ↓ |
| false response rate | gold IGNORE, predicted SPEAK or STOP | ↓ |
| respond rate | gold SPEAK, predicted SPEAK or STOP | ↑ |
| stop rate | gold STOP, predicted STOP | ↑ |
| latency | mean wall time per request across 16 concurrent requests on 4× H20 | ↓ |

### Baseline vs. Fine-Tuned Comparison

| Metric | Base (thinking off) | Base (thinking on) | LoRA r1 | LoRA r2 | LoRA r3 @800 (released) |
| :--- | ---: | ---: | ---: | ---: | ---: |
| accuracy ↑ | 0.581 | 0.445 | 0.806 | 0.857 | 0.885 |
| macro F1 ↑ | 0.374 | 0.380 | 0.707 | 0.810 | 0.737 |
| takeover rate ↓ | 0.023 | 0.088 | 0.089 | 0.155 | 0.081 |
| false response rate ↓ | 0.336 | 0.530 | 0.000 | 0.007 | 0.007 |
| respond rate ↑ | 0.394 | 0.301 | 0.735 | 0.811 | 0.770 |
| stop rate ↑ | 0.625 | 0.430 | 0.530 | 0.980 | 0.965 |
| parse failure ↓ | 0.001 | 0.343 | 0 | 0 | 0 |
| mean output tokens | 39.5 | 172.4 | 10.7 | 11.5 | 11.4 |
| latency (s) ↓ | 1.92 | 3.49 | 1.20 | 1.43 | 1.34 |

Experiment configurations:

- r1: 1,200 scenarios with 8 fields, trained for 3 epochs.
- r2: adds 1,482 scenarios containing topic switches, statements, and short stories, plus ASR-style noise.
- r3: adds mid-turn hard negatives, trained for 2 epochs with checkpoints saved every 200 steps.

### Per-Source Accuracy

| Source and gold label | n | Base | r1 | r2 | r3 @800 |
| :--- | ---: | ---: | ---: | ---: | ---: |
| candor_pause_handling (LISTEN) | 277 | 0.953 | 0.913 | 0.722 | 0.924 |
| synthetic_pause_handling (LISTEN) | 137 | 0.964 | 0.985 | 0.964 | 0.978 |
| synthetic_pause_handling (SPEAK) | 137 | 0.051 | 0.190 | 0.533 | 0.453 |
| icc_backchannel (BACKCHANNEL / LISTEN) | 315 | 0.984 | 0.768 | 0.813 | 0.886 |
| v1.5 user_interruption (STOP) | 200 | 0.625 | 0.530 | 0.980 | 0.965 |
| v1.5 user_backchannel (IGNORE) | 98 | 0.010 | 1.000 | 1.000 | 1.000 |
| v1.5 talking_to_other (IGNORE) | 100 | 0.000 | 1.000 | 1.000 | 1.000 |
| v1.5 background_speech (IGNORE) | 100 | 0.000 | 1.000 | 0.980 | 0.980 |
| v1.5 requests across subsets (SPEAK) | 498 | 0.43–0.67 | 0.74–0.99 | 0.76–1.00 | 0.76–0.99 |

### r3 Checkpoint Sweep

| Metric | @200 | @400 | @600 | @800 | @1000 |
| :--- | ---: | ---: | ---: | ---: | ---: |
| accuracy | 0.840 | 0.850 | 0.873 | 0.885 | 0.882 |
| macro F1 | 0.709 | 0.796 | 0.731 | 0.737 | 0.736 |
| takeover rate ↓ | 0.218 | 0.115 | 0.053 | 0.081 | 0.060 |
| respond rate ↑ | 0.833 | 0.753 | 0.701 | 0.770 | 0.735 |
| stop rate ↑ | 0.875 | 1.000 | 0.985 | 0.965 | 0.985 |

All evaluation reports are available in [`results/`](results/).

### Analysis

- Before fine-tuning, the base model over-triggers responses on this task and lacks turn-boundary discrimination. With thinking disabled, it responds to non-directed speech including side talk and background speech, resulting in a 33.6% false response rate and only 0.3% IGNORE recall, while failing to respond on most completed turns with a 39.4% respond rate. Its high accuracy on the CANDOR and ICC subsets stems from defaulting to LISTEN when the assistant state is silent rather than accurately identifying turn state.
- Disabling thinking yields better overall performance on this task. Enabling thinking increases average output length to 172 tokens and per-request latency to 3.49s, while 34.3% of outputs fail to emit an action tag within 128 tokens. Turn-taking decisions require low latency, so this recipe trains and serves with thinking disabled.
- Fine-tuning resolves non-directed speech filtering and interruption handling. Every fine-tuned run reaches nearly 100% accuracy across all three IGNORE subsets. Both r2 and r3@800 achieve at least a 96.5% stop rate on interruptions, while reducing average output length to roughly 11 tokens and cutting latency by 25% to 37%.
- The main trade-off is between end-of-turn response rate and mid-turn takeover rate:
  - r2 raised the end-of-turn respond rate to 0.811, but CANDOR pause accuracy dropped to 0.722 and the mid-turn takeover rate rose to 0.155.
  - Adding mid-turn hard negatives in r3 restored CANDOR accuracy to 0.924 and halved the takeover rate to 0.081, with only a small reduction in respond rate to 0.770.
  - The checkpoint sweep reflects the same trade-off: step 600 achieves the lowest takeover rate at 0.053, while step 200 achieves the highest respond rate at 0.833. We release step 800 for having the highest overall accuracy and a balanced trade-off between takeover and respond rates; choose step 400 or r2 if your application prioritizes macro F1 and responsiveness.
- Impact of hard negatives on BACKCHANNEL and strict macro F1: r3@800 outputs LISTEN rather than BACKCHANNEL on 279 of the 315 ICC examples. This is counted as correct under relaxed accuracy but as a miss under strict macro F1. Because BACKCHANNEL accounts for only 3.5% of the training set and the mid-turn hard negatives supervise the model to output LISTEN on similar unfinished contexts, the model favors LISTEN. Most remaining errors occur on finished turns in `synthetic_pause_handling` that are conservatively predicted as LISTEN.

## 8. Troubleshooting and Further Exploration

Based on the reproduction workflow and experimental findings of this recipe, below we summarize consumer hardware adaptation options and key tuning notes.

<a id="lightweight-hardware"></a>

### Lightweight Hardware Requirements

The original experiment used 8× H20 GPUs with multi-GPU parallelism and a micro-batch size of 16 to complete 2 epochs of 64,846 sample steps in about 80 minutes. Because each decision point has at most 512 prompt tokens and an average output of only 11.4 tokens, activation and KV cache footprints are small. You can reproduce the experiment on consumer GPUs or Apple Silicon Macs under either of the two configurations below:

- Hardware and changes for full-precision reproduction:
  - Hardware required: The BF16 base model takes about 15.8GB, the Rank-32 LoRA adapter takes about 1.18GB in BF16, and the short-sequence KV cache takes less than 0.5GB, so full-precision inference and evaluation require a Mac with at least 24GB unified memory or a single GPU with at least 24GB VRAM. Full-precision training additionally requires roughly 8.3GB for gradients and AdamW states across the 592M trainable parameters plus about 2GB for micro-batch activations, resulting in a peak memory footprint of 25GB to 27GB and requiring a Mac with at least 36GB unified memory or a single GPU with at least 32GB VRAM.
  - Required changes: For serving in `scripts/serve.sh`, set `--tp-size 1 --world-size 1`, specify `--backend mlx` on Mac, and lower `--max-running-prompts` to 2 or 4. For training in `train/train_lora.sh`, set `--tp-size 1 --world-size 1` and reduce `--mini-bs` from 16 to 1 or 2 while keeping `--batch-size 64` via gradient accumulation so the effective batch size and model precision remain unchanged.

- Minimum hardware and changes for lightweight reproduction:
  - Hardware required: Quantizing the model to 4-bit reduces weight memory to roughly 4.5GB, allowing inference and evaluation to run on an 8GB to 16GB unified memory Mac or an 8GB VRAM GPU. Training with 4-bit QLoRA at Rank 8 takes about 2.1GB for adapter parameters, gradients, and optimizer states plus about 2.5GB for single-example activations, keeping peak memory around 9GB to 10GB on a 16GB unified memory Mac or a 12GB to 16GB VRAM GPU.
  - Required changes: Add `--q4` to `areno train` to quantize the frozen base model to 4-bit at load time, lower the LoRA configuration to `--lora-rank 8 --lora-alpha 16`, pass the same rank and alpha when launching `areno serve`, and set `--batch-size 1 --mini-bs 1` while adding `--no-mlx-compile-train-step` on Mac MLX. Because the full `train.jsonl` contains 32,423 synthetic examples, subsample 3,000 to 5,000 stratified examples or pass `--max-steps 1000` on a single consumer machine to keep training time within 2 to 4 hours.

### Troubleshooting and Tuning Notes

- Pass `--lora-rank 32 --lora-alpha 64` explicitly when serving the fine-tuned adapter: `areno serve` defaults to a LoRA rank of 8, which will mismatch the rank-32 adapter if omitted.
- Why training rows are pre-tokenized: AReno's raw-text SFT path does not recognize Ling's role markers and would wrap the prompt twice, so `data/build_train.py` pre-tokenizes rows using the exact chat template rendered by `areno serve --disable-thinking`.
- Adjusting the BACKCHANNEL rate and takeover vs. responsiveness trade-off: if your application requires active backchannels or higher responsiveness, increase `P_BACKCHANNEL` in `data/build_train.py` or select the step 400 or r2 checkpoint.